<a href="https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook fills the contract for a refresh / content opportunity lane using the warehouse release. I have worked with a mid-panel month, March 2026, and I have kept the final month sealed for validation.

## 1. Unit of analysis + time window

1. One row means one content item at a decision point for refresh review.
2. I will use the warehouse fact table for daily performance and the content/client dimensions for joins and availability checks.
3. The feature window is March 2026, and the future window is April 2026.
4. I will predict a proxy label for whether that content item looks like a refresh opportunity: a decline proxy if April impressions fall below 80% of March impressions when March had non-zero visibility.
5. I deliberately exclude any field that comes from the future window or any product decision flag.

In [1]:
import os
import duckdb
import pandas as pd
from pathlib import Path


def load_hf_token():
    from google.colab import userdata

    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
        return HF_TOKEN
    except Exception:
        raise RuntimeError("HF_TOKEN not found in Colab Secrets.")


HF_TOKEN = load_hf_token()
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
march_path = f'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
april_path = f'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'

contract_sql = f"""
WITH march_base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_march,
        SUM(gsc_clicks) AS gsc_clicks_march,
        AVG(gsc_avg_position) AS gsc_avg_position_march,
        SUM(ga4_sessions) AS ga4_sessions_march,
        MAX(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS gsc_available_march,
        MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS ga4_available_march
    FROM read_parquet('{march_path}')
    GROUP BY 1, 2
),
april_base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_april
    FROM read_parquet('{april_path}')
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.gsc_impressions_march,
    m.gsc_clicks_march,
    m.gsc_avg_position_march,
    m.ga4_sessions_march,
    m.gsc_available_march,
    m.ga4_available_march,
    CASE
        WHEN a.gsc_impressions_april IS NULL THEN NULL
        WHEN m.gsc_impressions_march = 0 THEN 0
        WHEN a.gsc_impressions_april < 0.8 * m.gsc_impressions_march THEN 1 ELSE 0
    END AS future_decline_proxy
FROM march_base m
LEFT JOIN april_base a USING (client_hash_id, content_hash_id)
"""

contract_frame = con.execute(contract_sql).fetchdf()
contract_frame = contract_frame[contract_frame['future_decline_proxy'].notna()].copy()
contract_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions_march,gsc_clicks_march,gsc_avg_position_march,ga4_sessions_march,gsc_available_march,ga4_available_march,future_decline_proxy
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,1,1,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,1,0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,1,1,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,1,1,0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,1,1,0


## 2. Fields: feature / label / context / excluded

- Features: gsc_impressions_march, gsc_clicks_march, gsc_avg_position_march, ga4_sessions_march, gsc_available_march.
- Label / proxy: future_decline_proxy.
- Context: client_hash_id, content_hash_id, plus the monthly facts from March and April for joins and slicing.
- Excluded: April future metrics, trend_direction-style derived labels, and any product decision flags because they are not observed measurements from the decision moment.

In [2]:
feature_cols = ['gsc_impressions_march', 'gsc_clicks_march', 'gsc_avg_position_march', 'ga4_sessions_march', 'gsc_available_march']
label_col = 'future_decline_proxy'
context_cols = ['client_hash_id', 'content_hash_id']
excluded_cols = ['gsc_impressions_april', 'trend_direction', 'health_score']

bucket_map = {
    'features': feature_cols,
    'label_proxy': label_col,
    'context': context_cols,
    'excluded': excluded_cols,
}

pd.Series(bucket_map)

,0
features,"[gsc_impressions_march, gsc_clicks_march, gsc_..."
label_proxy,future_decline_proxy
context,"[client_hash_id, content_hash_id]"
excluded,"[gsc_impressions_april, trend_direction, healt..."


## 3. Verify it with queries (grain, counts, windows)

These three checks match the contract above and use the March 2026 slice.

In [3]:
print('Query 1 — grain check on the agreed unit of analysis')
q1 = f"""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_march,
        SUM(gsc_clicks) AS gsc_clicks_march,
        AVG(gsc_avg_position) AS gsc_avg_position_march,
        SUM(ga4_sessions) AS ga4_sessions_march,
        MAX(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS gsc_available_march,
        MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS ga4_available_march
    FROM read_parquet('{march_path}')
    GROUP BY 1, 2
)
SELECT client_hash_id, content_hash_id, COUNT(*) AS c
FROM base
GROUP BY 1, 2
HAVING COUNT(*) > 1
LIMIT 5
"""
print(con.execute(q1).fetchdf())

print('\nQuery 2 — slice row count and date span')
q2 = f"""
SELECT COUNT(*) AS rows_in_slice, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{march_path}')
"""
print(con.execute(q2).fetchdf())

print('\nQuery 3 — availability filter using IS TRUE')
q3 = f"""
SELECT COUNT(*) AS rows_with_gsc_and_ga4
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
"""
print(con.execute(q3).fetchdf())

Query 1 — grain check on the agreed unit of analysis


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [client_hash_id, content_hash_id, c]
Index: []

Query 2 — slice row count and date span


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_in_slice   min_date   max_date
0        9841378 2026-03-01 2026-03-31

Query 3 — availability filter using IS TRUE
   rows_with_gsc_and_ga4
0                 364347


## 4. Data limits

The main limitation is that this slice is biased toward content with usable search and analytics history. Early rows in the panel are GSC-only or sparse, so I cannot treat the whole history as equally complete.

In [4]:
limitation_sql = f"""
SELECT
    COUNT(*) AS rows_in_march,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_gsc,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4
FROM read_parquet('{march_path}')
"""
print(con.execute(limitation_sql).fetchdf())

   rows_in_march  rows_with_gsc  rows_with_ga4
0        9841378      3611061.0       413966.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.